# Lecture 02 अभ्यास: LoRA से उत्तर प्रारूप सिखाएँ और प्रशिक्षण से पहले तथा बाद की तुलना करें

काल्पनिक पाठ्यक्रम FAQ से एक छोटे, निर्देशों का पालन करने के लिए प्रशिक्षित मॉडल को थोड़े समय के लिए LoRA से प्रशिक्षित करेंगे और देखेंगे कि **क्या बदलता है और क्या नहीं**। मूल्यांकन में **प्रारूप पालन दर** (`answer` और `needs_review` वाला JSON) और **सामग्री की सही उत्तर दर** अलग रखेंगे।

**इस अभ्यास में क्या जाँचेंगे**

1. prompt में प्रारूप बताने पर भी छोटा मॉडल कभी उसका पालन नहीं करता (खंड 3, प्रशिक्षण से पहले)।
2. मूल जापानी डेटा वाले रन में कुछ मिनट LoRA से प्रशिक्षण के बाद प्रारूप लगभग हमेशा सही हुआ; अपने रन से तुलना करें (खंड 5)।
3. प्रारूप सही हो तब भी FAQ में अनुपस्थित प्रश्न पर निर्णय (पुष्टि माँगे या निश्चित उत्तर दे) की गारंटी नहीं है (खंड 5 और 6)।
4. उन्नत: उसी मॉडल को 4 बिट quantization से लोड करने पर प्रशिक्षण के बिना भी आउटपुट बदलता है (खंड 8)।

**क्रम**

| खंड | सामग्री | मूल जापानी डेटा वाले रन के आधार पर T4 का अनुमानित समय |
| --- | --- | --- |
| 0–3 | वातावरण, सेटिंग, डेटा, मॉडल और प्रशिक्षण से पहले मूल्यांकन | लगभग 5 मिनट |
| 4–7 | प्रशिक्षण, बाद का मूल्यांकन, ज्ञान प्रश्न, तुलना और जमा करना | लगभग 10 मिनट (4–5 मिनट प्रशिक्षण सहित) |
| 8 (वैकल्पिक) | 4 बिट quantization से लोड करके तुलना | लगभग 2 मिनट |

कक्षा से पहले खंड 0–3 चलाकर वातावरण जाँच सकते हैं (वैकल्पिक)। Colab सत्र टूटने पर चर मिट जाते हैं; उस स्थिति में Lecture के दिन खंड 0 से फिर चलाएँ।

**साधन**: Google Colab (T4 GPU, मुफ्त योजना)। मॉडल `Qwen/Qwen3.5-0.8B` (Apache-2.0) है। नोटबुक काल्पनिक डेटा स्वयं बनाती है; इसमें वास्तविक समय सीमाएँ या URL नहीं हैं।

**पढ़ने का तरीका**: हर खंड में "क्या करें", "क्या देखें" और "समस्या होने पर" दिया है। सेल ऊपर से नीचे चलाएँ। बीच का सेल छोड़ने पर आगे आवश्यक चर न मिलने की त्रुटि आएगी।

नीचे दिए समय, loss और आउटपुट संबंधी अवलोकन मूल जापानी FAQ और प्रश्नों पर किए गए अभ्यास से हैं। हिंदी में अनूदित FAQ और प्रश्नों पर माप तथा आउटपुट अलग हो सकते हैं; अपने रन के परिणाम दर्ज करें।

संस्करण: 1.1 (2026-10-18)


## 0. वातावरण जाँचें और लाइब्रेरी स्थापित करें

**क्या करें**: GPU मिला है या नहीं देखें और सत्यापित लाइब्रेरी संस्करण स्थापित करें।

**क्या देखें**

- पहले सेल में `Tesla T4` दिखे।
- दूसरे सेल के अंत में संस्करण सूची (`transformers: 5.18.0` आदि) दिखे। स्थापना में लगभग 1 मिनट लगता है।

**समस्या होने पर**

- "GPU नहीं मिला" दिखे: **Runtime → Change runtime type → T4 GPU** चुनकर सेल फिर चलाएँ। भीड़ के कारण GPU न मिले तो बाद में कोशिश करें।
- pip का लाल `ERROR: pip's dependency resolver ...` (`cudf` या `pyarrow` से जुड़ा) Colab में पहले से स्थापित दूसरे पैकेज से मतभेद की चेतावनी है। इस अभ्यास पर असर नहीं पड़ता; आगे बढ़ें।


In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU रनटाइम में nvidia-smi नहीं है
    gpu_info = ""
print(gpu_info or "GPU नहीं मिला। Runtime → Change runtime type → T4 GPU चुनें।")

In [ ]:
# Colab T4 (Python 3.13, torch 2.11.0+cu130) पर जाँचे संस्करण। Colab का torch ही इस्तेमाल करें
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 को transformers 5 चाहिए
    "peft": "0.21.1",           # LoRA का implementation
    "trl": "1.14.2",            # प्रशिक्षण लूप (SFTTrainer)
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab का torchao (0.10) peft 0.21 से असंगत है और प्रशिक्षण पर ImportError देता है। आवश्यक नहीं, इसलिए हटाएँ
%pip install -q {spec}

# cudf-cu13 और pyarrow की dependency चेतावनी ERROR दिखती है; cudf इस्तेमाल नहीं होगा, इसलिए अनदेखा करें
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. सेटिंग

**क्या करें**: मॉडल का नाम, प्रशिक्षण epochs (डेटा पर कितने चक्कर) आदि एक शब्दकोश `CONFIG` में रखें। यह खंड 7 में जमा होने वाली फ़ाइल में भी लिखा जाएगा, ताकि बाद में परीक्षण की स्थितियाँ पता रहें।

- `revision` निश्चित रखने से हर बार वही मॉडल वज़न मिलते हैं। इससे पहले-बाद और दूसरों से तुलना समान स्थितियों में होती है।
- `num_train_epochs: 20` का अर्थ 14 प्रशिक्षण उदाहरणों पर 20 चक्कर है। हर चक्कर 2 steps, कुल 40 steps; मूल जापानी डेटा वाले रन में T4 पर लगभग 4–5 मिनट लगे।
- `gen: {"do_sample": False}` greedy generation है। पहले-बाद की generation स्थितियाँ एक रखकर संयोगजन्य अंतर हटाते हैं।

`USE_DRIVE` को `True` करने पर Google Drive माउंट होकर प्रशिक्षित adapter और परिणाम फ़ाइलें वहाँ भी सहेजी जाती हैं। सत्र टूटने के बाद परिणाम चाहिए तो इसे इस्तेमाल करें। एक ही सत्र में काम पूरा हो तो `False` रहने दें।

**क्या देखें**: सेटिंग का JSON दिखेगा।


In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # निर्देशों का पालन करने के लिए प्रशिक्षित छोटा मॉडल (Apache-2.0)। बिना quantization के T4 में समाता है
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # हर रन में वही वज़न रखने के लिए निश्चित
    "seed": 42,
    "num_train_epochs": 20,  # 14 प्रशिक्षण उदाहरणों पर चक्कर; प्रति चक्कर 2 steps, कुल 40 steps (मूल जापानी डेटा वाले रन में T4 पर 4–5 मिनट)
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA मैट्रिक्स का आकार (छोटे मान पर कम पैरामीटर प्रशिक्षित होते हैं)
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # मूल्यांकन में greedy generation निश्चित रखें (पहले-बाद की स्थितियाँ समान)
    "work_dir": "/content/qlora_lab",                          # परिणाम फ़ाइलों का स्थान (बाईं फ़ाइल सूची में देखें)
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True पर सहेजने का स्थान
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True पर Google Drive माउंट करके adapter और आउटपुट सहेजें
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. काल्पनिक FAQ और प्रशिक्षण डेटा बनाएँ

**क्या करें**: पाँच काल्पनिक FAQ प्रविष्टियों से प्रशिक्षण (`train`), सत्यापन (`valid`) और मूल्यांकन (`test`) डेटा बनाएँ।

मॉडल का इनपुट "FAQ + प्रश्न" है और अपेक्षित आउटपुट `{"answer": <स्ट्रिंग>, "needs_review": <बूलियन>}` की एक JSON पंक्ति है।

- FAQ में प्रमाण वाला प्रश्न FAQ के पाठ से उत्तर पाए और `needs_review: false` रखे।
- FAQ में अनुपस्थित प्रश्न "यह जानकारी FAQ में नहीं है। आयोजकों से पुष्टि आवश्यक है।" लौटाए और `needs_review: true` रखे।

**डेटा का विभाजन**

- एक FAQ प्रविष्टि के लिए प्रश्नों के अलग शब्दांकन **template के आधार पर** train / valid / test में बाँटते हैं, ताकि प्रशिक्षण में देखा प्रश्न ज्यों का त्यों test में न आए।
- test में 21 प्रश्न हैं (15 प्रमाण वाले, 6 बिना प्रमाण)। संख्या पूरी करने के लिए केवल test में जाने वाले अलग प्रश्न हैं।
- सही उत्तर (`needs_review` का मान और उत्तर में होने वाले शब्द) `gold` में अलग रखते हैं और मॉडल को नहीं भेजते।
- train में 14 उदाहरण (10 प्रमाण वाले, 4 बिना प्रमाण) और valid में 7 हैं। FAQ में केवल पाँच प्रविष्टियाँ हैं, इसलिए डेटा कम है और वही उदाहरण बार-बार सीखे जाएँगे। दोहराव डेटा की प्रतिलिपियाँ बनाकर नहीं, खंड 4 के `num_train_epochs` से तय करें। डेटा फ़ाइल में उदाहरणों की प्रतिलिपियाँ न बनाएँ; प्रशिक्षण की मात्रा सेटिंग में रखें।

FAQ के फ़ाइलनाम `student_id_assignment_number.pdf` में `student_id` की जगह विद्यार्थी की पहचान संख्या और `assignment_number` की जगह असाइनमेंट संख्या रखें। शिक्षक से प्रश्न पूछने का निर्धारित समय (office hours) आगे "परामर्श समय" कहलाता है।

**क्या देखें**: `train=14 valid=7 test=21 gold=21` और एक उदाहरण (प्रश्न तथा अपेक्षित आउटपुट) दिखता है। train / valid / test में प्रश्न दोहराया जाए तो यहाँ त्रुटि आती है।


In [ ]:
FAQ = {
    "deadline": "असाइनमेंट जमा करने की अंतिम समय सीमा 30 सितंबर 23:59 (जापान समय) है।",
    "destination": "असाइनमेंट अध्ययन पोर्टल के असाइनमेंट अनुभाग में जमा करें।",
    "format": "एक PDF फ़ाइल जमा करें; उसका नाम student_id_assignment_number.pdf रखें।",
    "office_hours": "शिक्षक से प्रश्न पूछने का निर्धारित परामर्श समय हर मंगलवार 17:00–18:00 है।",
    "late": "अंतिम समय के 48 घंटे के भीतर देर से जमा करने पर अंक काटकर स्वीकार किया जाएगा।",
}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

# प्रमाण वाले: (FAQ key, प्रश्न templates, सही answer में आवश्यक शब्द)
GROUNDED = {
    "deadline": (["असाइनमेंट की अंतिम समय सीमा कब है", "कब तक जमा करना है", "कृपया अंतिम तारीख बताएँ", "जमा करने की समय सीमा क्या है?"], ["30 सितंबर", "23:59"]),
    "destination": (["कहाँ जमा करना है", "कहाँ अपलोड करना है", "असाइनमेंट कहाँ से जमा होगा", "जमा करने की जगह क्या है?"], ["अध्ययन पोर्टल", "असाइनमेंट"]),
    "format": (["जमा करने का प्रारूप क्या है", "फ़ाइल नाम का नियम क्या है", "किस फ़ाइल प्रारूप में जमा करें", "क्या PDF के अलावा कुछ और जमा कर सकते हैं"], ["PDF"]),
    "office_hours": (["प्रश्न कब पूछ सकते हैं", "परामर्श समय कब है", "शिक्षक से प्रश्न पूछने का समय है?", "सलाह के लिए समय क्या है?"], ["मंगलवार", "17:00"]),
    "late": (["देर से जमा करें तो क्या होगा", "समय सीमा के बाद स्वीकार होगा?", "देर से जमा करने का नियम क्या है?", "अंतिम समय के बाद क्या होगा"], ["48 घंटे", "अंक काटकर"]),
}
# बिना प्रमाण: FAQ में नहीं, इसलिए needs_review=true सही है
UNGROUNDED = [
    "क्या फिर से जमा कर सकते हैं", "किसी Lecture में अनुपस्थित रहने पर क्या होगा", "क्या दूसरे विद्यार्थी के अंक देख सकते हैं",
    "अंक कब आएँगे", "क्या अंतिम तारीख बढ़ सकती है", "क्या समूह में जमा कर सकते हैं",
    "असाइनमेंट के अंक कैसे बँटेंगे", "क्या Lecture की रिकॉर्डिंग उपलब्ध होगी",
]

# केवल test के अतिरिक्त प्रश्न (train / valid में नहीं)
TEST_EXTRA_GROUNDED = {
    "deadline": ["अंतिम तारीख और समय बताएँ", "असाइनमेंट कब और कितने बजे तक है"],
    "destination": ["असाइनमेंट कहाँ जमा होगा", "जमा करने का स्थान बताएँ"],
    "format": ["जमा की जाने वाली फ़ाइल किस प्रकार की है", "कितनी और किस प्रकार की फ़ाइल जमा करें"],
    "office_hours": ["परामर्श समय का दिन और समय बताएँ", "किस दिन और कब से प्रश्न पूछ सकते हैं"],
    "late": ["अंतिम समय तक जमा न करें तो क्या होगा", "क्या देर से जमा असाइनमेंट स्वीकार होगा"],
}
TEST_EXTRA_UNGROUNDED = [
    "क्या असाइनमेंट में अक्षरों की संख्या की कोई सीमा है?", "जमा काम पर प्रतिक्रिया मिलेगी?",
    "संदर्भ लिखने का कोई तय तरीका है?", "परामर्श समय के लिए पहले बुकिंग चाहिए?",
]
UNGROUNDED_ANSWER = "यह जानकारी FAQ में नहीं है। आयोजकों से पुष्टि आवश्यक है।"

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

def split_templates(templates):
    # पहले 2 templates train, तीसरा valid, चौथा test में रखें (एक ही शब्दांकन दो भागों में नहीं)
    return templates[:2], templates[2:3], templates[3:4]

train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]   # बिना प्रमाण के हर 4 प्रश्न: train 2 / valid 1 / test 1
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["पुष्टि"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["पुष्टि"]}

# train में 14 उदाहरण (10 प्रमाण वाले, 4 बिना प्रमाण)। उदाहरण कितनी बार सीखें, यह प्रतिलिपियाँ नहीं, खंड 4 के num_train_epochs से तय करें
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "train / valid / test में प्रश्न दोहराया गया है। templates जाँचें"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("एक प्रशिक्षण उदाहरण | प्रश्न:", train[0]["question"], "| अपेक्षित आउटपुट:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. मॉडल लोड करें और प्रशिक्षण से पहले मूल्यांकन करें

**क्या करें**: बेस मॉडल को quantization के बिना (32 बिट) लोड करें और test के 21 प्रश्न पूछकर अंकन करें।

prompt (`SYSTEM`) में "हमेशा एक पंक्ति में `answer` और `needs_review` वाला JSON उत्तर दें" लिखा है। इसलिए **प्रारूप का निर्देश पहले ही दिया जा चुका है**। प्रशिक्षण से पहले देखें कि मॉडल इसे मानता है या नहीं।

**अंकन कैसे होता है** (खंड 5 में भी यही फ़ंक्शन)

- **प्रारूप**: JSON पढ़ा जा सके; ठीक दो keys `answer` और `needs_review` हों; इनके प्रकार स्ट्रिंग और बूलियन हों।
- **सामग्री**: `needs_review` सही मान से मिले और अपेक्षित शब्द (`30 सितंबर` तथा `23:59` जैसे) `answer` में हों।

सामग्री सही माने जाने के लिए `needs_review` सही होना आवश्यक है। इसलिए key गायब हो तो पाठ सही होने पर भी सामग्री का परिणाम "गलत" होगा। तालिका में स्वयं उत्तर पढ़ें। प्रारूप की त्रुटियों के कोड का अर्थ है: `json_error` — JSON पढ़ने में त्रुटि; `not_object` — JSON ऑब्जेक्ट नहीं है; `extra_keys` — अतिरिक्त कुंजियाँ हैं; `missing_key` — आवश्यक कुंजी गायब है; `wrong_type` — मान का प्रकार गलत है।

`Qwen/Qwen3.5-0.8B` चित्र भी सँभाल सकता है, लेकिन `AutoModelForCausalLM` से लोड करने पर केवल पाठ वाला भाग इस्तेमाल होता है।

**क्या देखें**

- पहला सेल: `वज़न की मेमोरी (GB): 3.01` जैसा संदेश। मॉडल डाउनलोड (लगभग 1.75 GB) में लगभग 1 मिनट।
- दूसरा सेल: एक पंक्ति का सार (`format_rate`, `content_rate` आदि) और 21 प्रश्नों के आउटपुट। मूल्यांकन में लगभग 1 मिनट। मूल जापानी डेटा वाले रन में प्रशिक्षण से पहले बहुत से आउटपुट में `needs_review` अनुपस्थित था; अपने आउटपुट जाँचें।

**स्वीकार्य चेतावनियाँ**

- `You are sending unauthenticated requests to the HF Hub`: इसका अर्थ Hugging Face में login किए बिना मॉडल लेना है। सार्वजनिक मॉडल login के बिना डाउनलोड होते हैं। login की आवश्यकता इन स्थितियों में पड़ती है, मॉडल के आकार से इसका संबंध नहीं है:
  - उपयोग की शर्तें स्वीकारने वाले gated मॉडल (जैसे Llama, Gemma) या निजी रिपॉज़िटरी।
  - अपना मॉडल या डेटा Hub पर अपलोड करना।
  - थोड़े समय में इतने अनुरोध कि सीमा पूरी हो जाए। बिना login की प्रकाशित सीमा प्रति IP पते पर 5 मिनट में 3,000 फ़ाइल अनुरोध है (सितंबर 2025 का मान; [Hub rate limits](https://huggingface.co/docs/hub/rate-limits)); login पर सीमा बढ़ती है।

  इस नोटबुक का मॉडल सार्वजनिक है और सहमति नहीं माँगता। इसे लोड करने में अधिकतम कुछ दर्जन अनुरोध होते हैं, इसलिए सीमा तक नहीं पहुँचेंगे। यदि `429 Too Many Requests` आए भी तो लाइब्रेरी प्रतीक्षा समय पढ़कर स्वतः दोबारा प्रयास करती है। कभी login आवश्यक हो तो Colab के बाएँ ताले (Secrets) में Hugging Face token को `HF_TOKEN` नाम से दर्ज करके इस नोटबुक को पहुँच दें; token स्वतः इस्तेमाल होगा।
- `... is falling back to its reference PyTorch implementation ...`: मॉडल की कुछ linear attention परतों के लिए तेज़ implementation नहीं है, इसलिए धीमा लेकिन सही implementation चलता है। लगभग चार पंक्तियाँ दिखेंगी।

**यहाँ पढ़ें**: तालिका में देखें कि कौन सी keys गायब हैं और पाठ FAQ से मेल खाता है या नहीं। खंड 5 में इसे प्रशिक्षण के बाद वाले आउटपुट के साथ देखेंगे। मूल आउटपुट `/content/qlora_lab/eval_before.json` में भी सहेजा है (बाईं फ़ाइल सूची से खोलें)।


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# क्वांटाइज़ेशन के बिना। वज़न 32bit में ही रखें (dtype छोड़ने पर मॉडल मूल सटीकता में लोड होगा, इसलिए इसे स्पष्ट करें)
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("लोड किया गया मॉडल:", CONFIG["base_model"], "| डिवाइस:", DEVICE, "| वज़न की मेमोरी (GB):", BASE_MEMORY_GB)

# प्रॉम्प्ट में आउटपुट प्रारूप बताया गया है। प्रशिक्षण से पहले जाँचें कि मॉडल इसका पालन करता है या नहीं
SYSTEM = "आप पाठ्यक्रम के FAQ का उत्तर देने वाले सहायक हैं। केवल दिए गए FAQ के आधार पर हमेशा एक पंक्ति में JSON में उत्तर दें। प्रारूप: {\"answer\": स्ट्रिंग, \"needs_review\": बूलियन}। FAQ में जानकारी न हो तो answer में \"पुष्टि आवश्यक है\" लिखें और needs_review को true रखें।"

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\nप्रश्न: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: तर्क वाला ब्लॉक खाली रखकर केवल उत्तर जनरेट करें (मॉडल बदलने पर भी तुलना की शर्तें एक जैसी रहें)
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # मूल आउटपुट को बिना काटे दिखाएँ

def check_format(text: str):
    '''जाँचें कि JSON पढ़ा जा सकता है और आवश्यक कुंजियाँ व उनके प्रकार सही हैं; विफलता का कारण भी लौटाएँ'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''सामग्री की जाँच: needs_review सही उत्तर से मेल खाए और answer में आवश्यक शब्द हों'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "सही" if ok else "गलत"

def evaluate(model, label: str):
    '''test के 21 प्रश्नों के उत्तर जनरेट करके अंकित करें; सारांश और हर प्रश्न का परिणाम लौटाएँ। मूल आउटपुट eval_<label>.json में सहेजें'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # प्रारूप गलत हो, लेकिन JSON जैसा भाग पढ़ा जा सके, तो सामग्री की अलग से जाँच करें
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# हर प्रश्न का आउटपुट सही needs_review मान से मिलाएँ। देखें कि कौन सी कुंजी गायब है और उत्तर FAQ से मेल खाता है या नहीं
display(pd.DataFrame([{"प्रश्न": r["question"], "सही needs_review": r["gold_needs_review"], "प्रशिक्षण से पहले का आउटपुट": r["raw"],
                       "प्रारूप": mark(r["format_ok"]), "प्रारूप विफलता का कारण": r["format_reason"], "सामग्री": mark(r["content_ok"])} for r in before_rows]))

## 4. LoRA प्रशिक्षण (मूल जापानी डेटा वाले रन में T4 पर 4–5 मिनट)

### LoRA और QLoRA में अंतर

- **LoRA**: बेस मॉडल के वज़न स्थिर रखकर कुछ परतों में जोड़े छोटे मैट्रिक्स (`adapter`) को ही प्रशिक्षित करता है।
- **QLoRA**: बेस मॉडल के वज़न 4 बिट में quantize करके रखता है और उस पर समान रूप से केवल adapter प्रशिक्षित करता है। adapter स्वयं quantize नहीं होता।

QLoRA से बेस मॉडल के वज़न की मेमोरी घटती है। यह 16 बिट का लगभग चौथाई होती है, जिससे GPU में सीधे न समाने वाले बड़े मॉडल भी प्रशिक्षित हो सकते हैं (7 अरब पैरामीटर पर 16 बिट में लगभग 14 GB और 4 बिट में लगभग 3.5 GB)।

| पहलू | LoRA (बिना quantization) | QLoRA (4 बिट) |
| --- | --- | --- |
| वज़न मेमोरी | अधिक | कम (16 बिट का लगभग 1/4) |
| बेस मॉडल का आउटपुट | मूल जैसा | quantization की त्रुटि से बदल सकता है; प्रभाव मॉडल पर निर्भर है |
| प्रशिक्षण की गति | आधार | तेज़ नहीं होती; हर गणना पर 4 बिट वज़न वापस बदलने से अक्सर धीमी होती है |
| आवश्यक लाइब्रेरी | peft | peft और bitsandbytes (CUDA GPU आवश्यक) |

QLoRA लागू करते समय ध्यान दें:

- लोड करते समय `BitsAndBytesConfig(load_in_4bit=True, ...)` दें और LoRA जोड़ने से पहले `prepare_model_for_kbit_training(model)` चलाएँ।
- संग्रह की सटीकता (4 बिट) और गणना की सटीकता (`bnb_4bit_compute_dtype`, जैसे 16 बिट) अलग सेटिंग हैं।
- पहले-बाद की तुलना समान quantization वाले बेस मॉडल पर करें। पहले बिना quantization और बाद में 4 बिट से नापने पर दोनों प्रभाव मिल जाएँगे।
- adapter प्रशिक्षण के बेस मॉडल के अनुसार बनता है। बाँटते या दोबारा इस्तेमाल करते समय बेस मॉडल का ID, revision और quantization सेटिंग साथ लिखें।
- कुछ मॉडल के लिए 4 बिट प्रशिक्षण अनुशंसित नहीं है। Qwen3.5 में quantization त्रुटि सामान्य से अधिक होने के कारण [प्रशिक्षण लाइब्रेरी की मार्गदर्शिका](https://unsloth.ai/docs/models/qwen3.5/fine-tune) 4 बिट प्रशिक्षण से मना करती है।

इस अभ्यास का मॉडल बिना quantization के T4 में समाता है, इसलिए LoRA से प्रशिक्षित करेंगे। पहले देखें मॉडल GPU में समाता है या नहीं; न समाए तो QLoRA या छोटा मॉडल चुनें। 4 बिट लोड से मेमोरी और आउटपुट में अंतर खंड 8 (उन्नत) में देख सकते हैं।

### प्रशिक्षण चलाएँ

**क्या करें**: train के 14 उदाहरण chat प्रारूप (system / user / assistant) में बदलें, LoRA जोड़ें और 20 epochs (40 steps) प्रशिक्षित करें। adapter को `adapter/` में सहेजें।

- `target_modules` वे परतें हैं जहाँ LoRA लगता है। इसमें attention projection (`q_proj` आदि) और MLP projection (`gate_proj` आदि) हैं। इस मॉडल की 24 परतों में 18 linear attention वाली हैं; शेष 6 में ही `q_proj` आदि हैं। MLP सभी परतों में है।
- batch 4 × gradient accumulation 2 = प्रति step 8 उदाहरण। train में 14 हैं, इसलिए एक epoch में 2 steps (8 और 6 उदाहरण) और 20 epochs में 40 steps। हर उदाहरण ठीक 20 बार सीखा जाता है। learning rate का warmup और cosine decay इन्हीं 40 steps पर लागू होता है।
- प्रशिक्षित मॉडल `tuned` चर में रहता है। LoRA बेस मॉडल की परतों में जुड़ता है, इसलिए पुराने `model` चर से generation पर भी adapter का असर रहता है। पहले के आउटपुट के लिए खंड 6 की तरह adapter अस्थायी रूप से बंद करें।

**क्या देखें**

- `trainable params: ... || all params: ... || trainable%: ...`: कुल पैरामीटरों का केवल कुछ प्रतिशत प्रशिक्षित होना LoRA की विशेषता है।
- `Training Loss` तालिका (हर 10 steps): मूल जापानी डेटा वाले रन में loss लगभग 1.5 से शुरू होकर 40 steps पर 0.01–0.02 तक घटा। अपने रन में loss का बदलाव दर्ज करें; अलग मान अपने आप में सेटिंग की त्रुटि नहीं बताते। यदि प्रशिक्षण पूरा होने तक loss नहीं घटता, तो जाँचें कि सेटिंग बदली तो नहीं है।
- `प्रशिक्षण समय: ... सेकंड | चरणों की संख्या: 40 | एपॉक: 20.0`: मूल जापानी डेटा वाले रन में T4 पर 240–290 सेकंड लगे। चरणों की संख्या और एपॉक पुष्टि करते हैं कि 14 उदाहरण × 20 चक्कर चले।

**स्वीकार्य चेतावनी**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` बताती है कि अंतिम टोकन की सेटिंग tokenizer से मिलाई गई है।

**समस्या होने पर**: प्रशिक्षण बीच में रुके या चल न पाए तो नीचे 4b पढ़ें।


In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # प्रशिक्षण डेटा में इनपुट (system + user) और अपेक्षित आउटपुट (assistant) की जोड़ियाँ हैं। मूल्यांकन वाला ही build_messages इस्तेमाल करें
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# Qwen3.5 की 24 परतों में से 18 में linear attention है; q/k/v/o_proj केवल शेष 6 परतों में हैं। gate/up/down_proj (MLP) सभी परतों में हैं
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # आम तौर पर इसे न बदलें। True करने पर केवल गणना 16bit मिश्रित सटीकता में होगी (सिर्फ़ CUDA पर; वज़न 32bit में रहेंगे)। इससे गति बढ़ सकती है, लेकिन loss nan भी हो सकता है
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA से प्रशिक्षित होने वाले पैरामीटरों का अनुपात
t0 = time.time()
trainer.train()  # बीच में रुकने पर फिर शुरू करने के लिए खंड 4b देखें
print(f"प्रशिक्षण समय: {time.time()-t0:.0f} सेकंड | चरणों की संख्या: {trainer.state.global_step} | एपॉक: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # प्रशिक्षण मोड में dropout सक्रिय रहता है और मूल्यांकन के आउटपुट बदल सकते हैं (`_` में रखने से पूरा मॉडल स्क्रीन पर नहीं दिखता)

### 4b. प्रशिक्षण रुक गया या नहीं हो पाया

**बीच में रुका (जैसे रनटाइम डिस्कनेक्ट)**: checkpoint हर 10 steps पर `/content/qlora_lab/ckpt/` में सहेजा जाता है। फ़ाइल बची हो तो खंड 0–3 फिर चलाएँ। फिर ऊपर के प्रशिक्षण सेल में `trainer.train()` को `trainer.train(resume_from_checkpoint=True)` से बदलकर चलाएँ; अंतिम checkpoint से 40 steps (20 epochs) तक जारी रहेगा। पूरा रनटाइम मिट गया और फ़ाइल न हो तो फिर शुरू करें या नीचे का विकल्प चुनें।

**प्रशिक्षण नहीं हो पाया (GPU उपलब्ध नहीं या समय कम)**: प्रशिक्षक ने मूल जापानी डेटा वाली नोटबुक T4 पर चलाई है। उनके परिणाम (प्रशिक्षण के बाद के आउटपुट और अंक) से तुलना और विचार करें। खंड 5 से आगे प्रशिक्षित मॉडल `tuned` के बिना कोड सेल नहीं चलेंगे; प्रशिक्षक के परिणाम के साथ विवरण पढ़ें। जमा करते समय खंड 7 में `executed_training_myself` को `False` करें और स्वयं कहाँ तक चलाया, साफ़ लिखें।


## 5. प्रशिक्षण के बाद मूल्यांकन (वही test और generation सेटिंग)

**क्या करें**: प्रशिक्षित मॉडल `tuned` को पहले जैसे 21 प्रश्न, उन्हीं generation सेटिंग से जाँचें और परिणाम साथ रखें। केवल adapter की उपलब्धता बदली है।

**क्या देखें**

- तुलना तालिका: प्रशिक्षण से पहले और बाद `format_rate` (प्रारूप पालन दर) तथा `content_rate` (सामग्री सही उत्तर दर) कैसे बदले।
- `गिरावट`: पहले प्रारूप और सामग्री दोनों सही थे, बाद में गलत हुए प्रश्न। सूची खाली होना वांछनीय है; अपने रन में दिखने वाली गिरावट की जाँच करें।
- प्रश्नों की तालिका में बची त्रुटियाँ और उनके प्रकार पढ़ें:
  - बिना प्रमाण वाले प्रश्न (सही `needs_review` `True`) पर FAQ की दूसरी प्रविष्टि लगाकर निश्चित उत्तर दिया।
  - प्रमाण वाले प्रश्न (सही मान `False`) पर "दर्ज नहीं है" कहकर पुष्टि की माँग से टाल दिया।

मूल जापानी डेटा वाले रन में प्रशिक्षण के बाद उत्तर प्रशिक्षण डेटा में FAQ के मूल वाक्य के लगभग समान थे। अपने रन में भी यह तुलना करें। प्रारूप और वाक्य ठीक हो सकते हैं, पर "FAQ में प्रमाण है या नहीं" का निर्णय सीखा गया हो, यह निश्चित नहीं है।

seed तय होने पर भी GPU प्रशिक्षण पूरी तरह दोहराया नहीं जा सकता। प्रारूप और सामग्री की दरें लगभग समान हों तब भी विफल प्रश्न हर रन में अलग हो सकते हैं। सहपाठी के "बचे हुए प्रश्न" अलग हों तो सामान्य है; त्रुटियों के प्रकार की तुलना करें।


In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# वे प्रश्न जो प्रशिक्षण से पहले प्रारूप और सामग्री दोनों में सही थे, लेकिन बाद में गलत हुए (गिरावट)
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("गिरावट:", regressed)

# हर प्रश्न का मूल आउटपुट देखें। प्रारूप (JSON, आवश्यक कुंजियाँ, प्रकार) और सामग्री (needs_review का मिलान और आवश्यक शब्द) की जाँच स्वयं करें
detail = pd.DataFrame([{"प्रश्न": b["question"], "सही needs_review": b["gold_needs_review"],
                        "प्रशिक्षण से पहले का आउटपुट": b["raw"], "प्रशिक्षण से पहले प्रारूप/सामग्री": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "प्रशिक्षण के बाद का आउटपुट": a["raw"], "प्रशिक्षण के बाद प्रारूप/सामग्री": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. ज्ञान जाँच प्रश्न (प्रारूप सीखने से ज्ञान की गारंटी नहीं)

**क्या करें**: प्रारूप प्रयोग से अलग, FAQ और प्रशिक्षण डेटा दोनों में अनुपस्थित तथ्यों पर तीन प्रश्न पूछें (तुलना के लिए FAQ में मौजूद एक प्रश्न भी)। अपेक्षित उत्तर "पुष्टि आवश्यक है (`needs_review: true`)" है।

उत्तर तीन श्रेणियों में रखें:

| वर्गीकरण | अर्थ |
| --- | --- |
| `format_error` | JSON प्रारूप गलत (प्रशिक्षण से पहले आम) |
| `needs_review` | "दर्ज नहीं है; पुष्टि आवश्यक है" लौटाया (अपेक्षित उत्तर) |
| `asserted` | `needs_review: false` के साथ निश्चित उत्तर दिया; FAQ में अनुपस्थित प्रश्न पर यह बिना प्रमाण का दावा है |

**क्या देखें**: मूल जापानी अभ्यास के विवरण में प्रशिक्षण से पहले प्रारूप त्रुटियाँ आम बताई गई हैं। प्रशिक्षण के बाद "पुष्टि आवश्यक है" और "निश्चित उत्तर" दोनों मिल सकते हैं। अपने रन में देखें कि कौन से परिणाम मिले। तालिका में मूल आउटपुट पढ़ें और निश्चित उत्तर पर देखें कि मॉडल ने कौन सा कथित आधार (किसी दूसरी FAQ प्रविष्टि का उपयोग?) लिया। "फ़ाइन ट्यूनिंग से ज्ञान कभी सुधरता ही नहीं" पहले से न मानें; इसी डेटा के परिणाम लिखें।

प्रशिक्षण से पहले के आउटपुट adapter अस्थायी रूप से बंद करके (`tuned.disable_adapter()`) लिए जाते हैं।


In [ ]:
KNOWLEDGE_QUESTIONS = [
    "इस पाठ्यक्रम के अंतिम असाइनमेंट की समय सीमा कब है",       # FAQ में नहीं है (काल्पनिक)
    "पाठ्यक्रम के शिक्षक का नाम क्या है",                     # FAQ में नहीं है
    "अगले Lecture का विषय क्या है",                 # FAQ में नहीं है
    "जमा करने की समय सीमा क्या है?",                   # FAQ में है (तुलना के लिए)
]
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

grounded_questions = {ex['question'] for ex in train + valid + test if not ex['needs_review']}
knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # LoRA आधार मॉडल की परतों में जुड़ता है, इसलिए प्रशिक्षण के बाद `model` से जनरेट करने पर भी adapter शामिल होगा। प्रशिक्षण से पहले का आउटपुट पाने के लिए adapter अस्थायी रूप से बंद करें
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"प्रश्न": q, "FAQ में है": q in grounded_questions, "प्रशिक्षण से पहले वर्ग": classify(before_raw), "प्रशिक्षण से पहले का आउटपुट": before_raw,
                 "प्रशिक्षण के बाद वर्ग": classify(after_raw), "प्रशिक्षण के बाद का आउटपुट": after_raw})
display(pd.DataFrame(knowledge_rows))

## 7. तुलना तालिका और जमा करना

**क्या करें**: सेटिंग, लाइब्रेरी संस्करण, प्रशिक्षण से पहले और बाद के सारांश और बिगड़े प्रश्न एक `result.json` में रखें।

**क्या जमा करें**

- `result.json`, `eval_before.json`, `eval_after.json` (`/content/qlora_lab` में हैं; बाईं फ़ाइल सूची से दायाँ क्लिक → Download। `USE_DRIVE = True` हो तो Drive में भी प्रतिलिपि)।
- खंड 5 की तुलना तालिका और बची त्रुटियों के प्रकार का विवरण।
- खंड 6 के परिणाम (पुष्टि माँगने लगे उत्तर और अभी भी निश्चित दावे वाले उत्तर)।
- आपने स्वयं कितना प्रशिक्षण चलाया। प्रशिक्षक के परिणाम इस्तेमाल किए तो `executed_training_myself` को `False` रखें।


In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA (बिना क्वांटाइज़ेशन)",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # शिक्षक के परिणाम इस्तेमाल किए हों तो False करें
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive में सहेजा:", CONFIG["drive_dir"])

## 8. उन्नत: 4 बिट quantization से लोड करके मेमोरी और आउटपुट देखें (वैकल्पिक)

**क्या करें**: उसी मॉडल को 4 बिट (NF4) में लोड करके देखें कि वज़न की मेमोरी और प्रशिक्षण से पहले के आउटपुट कैसे बदलते हैं। QLoRA इसी 4 बिट बेस मॉडल में LoRA जोड़कर प्रशिक्षण करता है। यहाँ प्रशिक्षण नहीं होगा।

- bitsandbytes को CUDA GPU चाहिए, इसलिए GPU रनटाइम में ही चलेगा। स्थापना में लगभग 1 मिनट।
- 4 बिट लोड करते समय `device_map="auto"` मॉडल को GPU पर रखता है (bitsandbytes की लोड विधि के अनुसार)।
- 4 बिट पर प्रशिक्षण आज़माएँ तो पहले `prepare_model_for_kbit_training(model_4bit)` चलाएँ, फिर खंड 4 की प्रक्रिया अपनाएँ। पहले और बाद दोनों में 4 बिट बेस मॉडल से मूल्यांकन करें।

**क्या देखें**

- वज़न मेमोरी: 32 बिट पर लगभग 3 GB के मुकाबले 4 बिट पर 1 GB से कम।
- तुलना और प्रश्न तालिका: प्रशिक्षण नहीं हुआ है, इसलिए 32 बिट प्रशिक्षण से पहले (`before`) का अंतर quantization का असर है। कुछ प्रश्न सुधर सकते हैं; कुछ में `needs_review` का मान `"false"` (स्ट्रिंग) होकर प्रकार जाँच में विफल हो सकता है, या बिना प्रमाण के प्रश्न पर "हाँ" / "नहीं" का निश्चित उत्तर मिल सकता है। देखें कि प्रभाव की दिशा एक जैसी नहीं है।


In [ ]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # संग्रहण सटीकता: 4 बिट (NF4)
                         bnb_4bit_compute_dtype=torch.float16,             # गणना सटीकता: 16 बिट (T4 में bf16 समर्थित नहीं है)
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("वज़न की मेमोरी (GB) | बिना क्वांटाइज़ेशन (32 बिट):", BASE_MEMORY_GB, "| 4 बिट:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# इस मॉडल को प्रशिक्षित नहीं किया गया है। इसलिए बिना क्वांटाइज़ेशन वाले प्रशिक्षण से पहले के मॉडल (before) से अंतर, क्वांटाइज़ेशन का प्रभाव है
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"प्रश्न": r["question"], "सही needs_review": r["gold_needs_review"], "4 बिट, प्रशिक्षण से पहले का आउटपुट": r["raw"],
                       "प्रारूप/सामग्री": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "प्रारूप विफलता का कारण": r["format_reason"]} for r in q_rows]))